---

<h1 style="text-align: center;"><strong>Data Science & Statistical Computing</strong></h1>
<h2 style="text-align: center;"><strong>Aula 02 - Medidas Estatísticas</strong></h2>
<h3 style="text-align: center;"><strong>Prof. Jones Egydio</strong></h3>
<h4 style="text-align: center;"><strong>FIAP - 2026</strong></h4>
<br>
<br>



---

# **Aula 2: Medidas Estatísticas**

## **Objetivos**
- Compreender e calcular medidas de localização (tendência central e posição).
- Compreender e calcular medidas de variabilidade (dispersão).
- Identificar outliers com regras simples e interpretar o impacto deles.
- Aplicar os conceitos em um dataset público usando Python (pandas, numpy e scipy).

## **Dataset**
Vamos usar o dataset iris, o mesmo utilizado na Aula 01. Ele contém medidas de flores (comprimento e largura de sépala e pétala) para três espécies de iris. O objetivo aqui não é classificar, mas usar as colunas numéricas para explorar medidas descritivas.

In [1]:
# imports e carregamento do dataset iris
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()

# renomeando para ficar mais didático
df = df.rename(columns={
    "sepal length (cm)": "sepal_length_cm",
    "sepal width (cm)": "sepal_width_cm",
    "petal length (cm)": "petal_length_cm",
    "petal width (cm)": "petal_width_cm",
    "target": "species_id"
})

# mapeando id -> nome da espécie
df["species"] = df["species_id"].map(dict(enumerate(iris.target_names)))

df.head()

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species_id,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


---

## **1 Medidas Estatísticas**
Nesta aula, vamos trabalhar com duas grandes famílias de medidas descritivas: medidas de localização (onde os dados se concentram) e medidas de variabilidade (o quanto os dados se espalham). Também vamos ver um conceito essencial em análise exploratória: outliers.

## **1.1 Estimativas de localização**
Estimativas de localização resumem um conjunto de valores com um número representativo do centro (como média e mediana) ou da posição (como percentis). Elas ajudam a comparar grupos, descrever uma distribuição e apoiar decisões baseadas em dados.

### **1.1.1 Média**
A média aritmética é a soma dos valores dividida pela quantidade de observações. Ela é uma medida de tendência central muito usada quando a distribuição é aproximadamente simétrica e não há valores extremos dominando o comportamento. Em termos de interpretação, a média responde qual valor único preserva o total quando substituímos todos os valores por ele.

**Fórmula:**
Seja um conjunto de valores $x_1, x_2, \dots, x_n$. A média aritmética é:
$$
\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

In [2]:
# exemplo: média do comprimento da pétala (cm)
col = "petal_length_cm"
media = df[col].mean()
media

np.float64(3.7580000000000005)

#### NOTA:
Aqui foi tirada a média ARITMÉTICA de uma coluna inteira, usando o .mean()

### **1.1.2 Média ponderada**
A média ponderada é uma generalização da média em que cada observação recebe um peso. Ela é útil quando alguns valores devem ter maior influência, como notas com pesos diferentes, preços com quantidades diferentes ou métricas agregadas por volume.

**Fórmula:**
Seja $w_i$ o peso associado a $x_i$. A média ponderada é:
$$
\bar{x}_w = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}
$$

Observação: este exemplo usa pesos artificiais apenas para fins didáticos. Em problemas reais, os pesos vêm do contexto, como frequência, volume, relevância ou confiabilidade.

In [4]:
# exemplo: média ponderada do comprimento da pétala, dando mais peso a uma espécie
# aqui vamos criar pesos maiores para "virginica" para simular um cenário de importância diferente
col = "petal_length_cm"
weights = np.where(df["species"] == "virginica", 3, 1)  # virginica pesa 3, outras pesam 1

media_ponderada = np.average(df[col].values, weights=weights)
media_ponderada

np.float64(4.4756)

### **1.1.3 Mediana**
A mediana é o valor que divide os dados ao meio quando eles estão ordenados. Metade das observações fica abaixo ou igual e metade fica acima ou igual. Ela é mais robusta a outliers do que a média.

**Fórmula:**
Considere os dados ordenados $x_{(1)} \le \dots \le x_{(n)}$.
- Se $n$ é ímpar, a mediana é $x_{(\frac{n+1}{2})}$.
- Se $n$ é par, a mediana é $\frac{x_{(\frac{n}{2})} + x_{(\frac{n}{2}+1)}}{2}$.

In [5]:
# exemplo: mediana do comprimento da pétala (cm)
col = "petal_length_cm"
mediana = df[col].median()
mediana

np.float64(4.35)

In [6]:
col = "petal_length_cm"

s = df[col].sort_values().reset_index(drop=True)
n = len(s)
mid = n // 2

tabela = pd.DataFrame({col: s})
tabela["marcador"] = ""

if n % 2 == 1:
    tabela.loc[mid, "marcador"] = "<- mediana"
else:
    tabela.loc[[mid-1, mid], "marcador"] = "<- centro"

tabela.iloc[max(mid-10, 0): min(mid+11, n)]

,petal_length_cm,marcador
65,4.0,
66,4.1,
67,4.1,
68,4.1,
69,4.2,
70,4.2,
71,4.2,
72,4.2,
73,4.3,
74,4.3,<- centro


### **1.1.4 Moda**
A moda é o valor mais frequente em um conjunto de dados. Ela é especialmente útil em dados categóricos, como a categoria mais comum. Em dados numéricos contínuos, a moda pode não ser tão informativa quando quase todos os valores são únicos.

**Fórmula:**
A moda $m$ é o valor que maximiza a frequência:
$$
m = \arg\max_x \; \text{freq}(x)
$$
Quando há mais de um valor com a mesma maior frequência, existem múltiplas modas.

In [7]:
# exemplo: moda do "species" (categórico)
moda_species = df["species"].mode()
moda_species

0        setosa
1    versicolor
2     virginica
Name: species, dtype: str

### **1.1.5 Média aparada**
A média aparada é calculada removendo uma fração dos menores e dos maiores valores antes de calcular a média. Ela reduz o efeito de outliers e pode ser uma alternativa entre média e mediana.

**Fórmula:**
Seja $\alpha$ a proporção aparada em cada cauda (por exemplo, $\alpha=0{,}10$). Após ordenar os dados, removem-se os $\alpha n$ menores e os $\alpha n$ maiores valores. A média aparada é:
$$
\bar{x}_{\text{trim}} = \frac{1}{n-2k}\sum_{i=k+1}^{n-k} x_{(i)}
$$
onde $k=\lfloor \alpha n \rfloor$ e $x_{(i)}$ são os dados ordenados.

In [8]:
# exemplo: média aparada de 10% para o comprimento da pétala
from scipy.stats import trim_mean

col = "petal_length_cm"
media_aparada_10 = trim_mean(df[col].values, proportiontocut=0.10)
media_aparada_10

np.float64(3.7600000000000002)

### **1.1.6 Percentil**
Percentis são medidas de posição. O percentil p é um valor tal que aproximadamente p% das observações ficam abaixo ou igual a esse valor. Percentis ajudam a entender a distribuição e também são base para regras de detecção de outliers, como a regra do iqr.

**Fórmula:**
O percentil $p$ corresponde ao quantil $q=p/100$. Um modo de representar é:
$$
P_p = Q(q)
$$
Em termos práticos, $P_{50}$ corresponde à mediana, $P_{25}$ ao primeiro quartil $Q_1$ e $P_{75}$ ao terceiro quartil $Q_3$.

In [9]:
# exemplo: percentis 25, 50 e 75 do comprimento da pétala
col = "petal_length_cm"
p25 = df[col].quantile(0.25)
p50 = df[col].quantile(0.50)
p75 = df[col].quantile(0.75)

p25, p50, p75

(np.float64(1.6), np.float64(4.35), np.float64(5.1))

---

## **1.2 Definição de outliers**
Outliers são observações que se destacam por estarem muito distantes do padrão do restante dos dados. Eles podem surgir por erro de medição, erro de digitação, problemas de coleta ou podem ser casos reais e raros que são importantes.

Um método prático e muito usado em análise exploratória é a regra do iqr (intervalo interquartil).

**Fórmulas:**
$$
Q_1 = P_{25}, \quad Q_3 = P_{75}, \quad IQR = Q_3 - Q_1
$$
$$
L_{\text{inf}} = Q_1 - 1{,}5\cdot IQR, \quad L_{\text{sup}} = Q_3 + 1{,}5\cdot IQR
$$
Valores fora de $[L_{\text{inf}}, L_{\text{sup}}]$ são candidatos a outliers.

In [ ]:
# exemplo: detectando outliers por iqr no comprimento da pétala
col = "petal_length_cm"

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

outliers = df[(df[col] < lim_inf) | (df[col] > lim_sup)][[col, "species"]]

q1, q3, iqr, lim_inf, lim_sup, outliers.head(10)

In [ ]:
# visualização simples: boxplot para enxergar possíveis outliers
import matplotlib.pyplot as plt

col = "petal_length_cm"

plt.figure(figsize=(6, 2.5))
plt.boxplot(df[col].values, vert=False)
plt.title("boxplot - petal_length_cm")
plt.xlabel("cm")
plt.show()

---

## **1.3 Estimativas de variabilidade**
Estimativas de variabilidade quantificam o espalhamento dos dados. Duas distribuições podem ter a mesma média, mas dispersões muito diferentes, o que muda a interpretação.

Nesta seção, vamos ver medidas clássicas e medidas baseadas em desvios absolutos, que tendem a ser mais robustas em cenários com outliers.

### **1.3.1 Desvio padrão**
O desvio padrão mede, em média, o quão distantes os valores estão do centro na mesma unidade dos dados. Ele é a raiz quadrada da variância. Quanto maior o desvio padrão, maior a dispersão.

**Fórmula:**
O desvio padrão populacional é:
$$
\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i-\mu)^2}
$$
O desvio padrão amostral é:
$$
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}
$$

In [ ]:
# exemplo: desvio padrão amostral do comprimento da pétala (cm)
col = "petal_length_cm"
desvio_padrao = df[col].std(ddof=1)  # ddof=1 -> amostra
desvio_padrao

### **1.3.2 graus de liberdade**
Graus de liberdade são a quantidade de informação livre após impor restrições em um cálculo estatístico. No caso da variância amostral, usamos n−1 no denominador porque a média amostral foi estimada a partir dos dados, o que consome um grau de liberdade.

**Fórmula:**
Quando calculamos a variância amostral, usamos:
$$
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2
$$
O termo $n-1$ reflete os graus de liberdade. Em Python, isso aparece como `ddof`:
- `ddof=0` usa $n$.
- `ddof=1` usa $n-1$.

In [ ]:
# exemplo: comparando variância e desvio padrão com ddof=0 (população) e ddof=1 (amostra)
col = "petal_length_cm"
n = df[col].shape[0]

var_pop = df[col].var(ddof=0)
var_amostra = df[col].var(ddof=1)

std_pop = df[col].std(ddof=0)
std_amostra = df[col].std(ddof=1)

n, var_pop, var_amostra, std_pop, std_amostra

### **1.3.3 Variância**
A variância mede o espalhamento dos dados a partir dos desvios ao quadrado em relação à média. Ela é central em muitos métodos estatísticos, mas está em unidade ao quadrado, o que pode dificultar a interpretação direta. O desvio padrão é a raiz da variância e volta para a unidade original.

**Fórmula:**
A variância populacional é:
$$
\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i-\mu)^2
$$
A variância amostral é:
$$
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2
$$

In [ ]:
# exemplo: variância amostral do comprimento da pétala
col = "petal_length_cm"
variancia = df[col].var(ddof=1)
variancia

### **1.3.4 Amplitude**
A amplitude é a diferença entre o maior e o menor valor do conjunto. Ela é simples e intuitiva, mas é muito sensível a outliers, porque depende apenas dos extremos.

**Fórmula:**
A amplitude $A$ é:
$$
A = \max(x) - \min(x)
$$

In [ ]:
# exemplo: amplitude do comprimento da pétala
col = "petal_length_cm"
amplitude = df[col].max() - df[col].min()
amplitude

### **1.3.5 Desvio absoluto médio**
O desvio absoluto médio mede a média das distâncias absolutas dos valores em relação a um centro. Quando o centro é a média, ele resume a dispersão com menos sensibilidade a valores extremos do que a variância, pois não eleva desvios ao quadrado.

**Fórmula:**
Usando a média como centro, o desvio absoluto médio é:
$$
DAM = \frac{1}{n}\sum_{i=1}^{n} |x_i-\bar{x}|
$$

In [ ]:
# exemplo: desvio absoluto médio em torno da média
col = "petal_length_cm"
x = df[col].values

dam = np.mean(np.abs(x - np.mean(x)))
dam

### **1.3.6 Desvio absoluto mediano da mediana**
O desvio absoluto mediano da mediana, conhecido como mad robusto, é a mediana de |x − mediana(x)|. Ele é uma medida robusta de dispersão, pois usa medianas e é muito menos influenciado por outliers.

**Fórmula:**
Defina $m = \text{mediana}(x)$. O mad robusto é:
$$
MAD = \text{mediana}\left(|x_i - m|\right)
$$
Em algumas aplicações, usa-se uma constante de escala para aproximar o desvio padrão sob normalidade, mas aqui vamos usar a forma direta para interpretação.

In [ ]:
# exemplo: mad robusto em torno da mediana
col = "petal_length_cm"
x = df[col].values

med = np.median(x)
mad_robusto = np.median(np.abs(x - med))
mad_robusto

---

## **Encerramento**
Nesta aula, você aplicou medidas de localização, variabilidade e uma regra prática para outliers em um dataset público. Na sequência, você pode repetir os mesmos cálculos por grupo para comparar distribuições e praticar segmentação de dados.

In [10]:
# exercício rápido: calcule as principais medidas por espécie
cols_num = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]

resumo_por_especie = (
    df.groupby("species")[cols_num]
      .agg(["count", "mean", "median", "std", "min", "max"])
)

resumo_por_especie

sepal_length_cm                                   sepal_width_cm  \
                     count   mean median       std  min  max          count   
species                                                                       
setosa                  50  5.006    5.0  0.352490  4.3  5.8             50   
versicolor              50  5.936    5.9  0.516171  4.9  7.0             50   
virginica               50  6.588    6.5  0.635880  4.9  7.9             50   

                                    ... petal_length_cm                      \
             mean median       std  ...          median       std  min  max   
species                             ...                                       
setosa      3.428    3.4  0.379064  ...            1.50  0.173664  1.0  1.9   
versicolor  2.770    2.8  0.313798  ...            4.35  0.469911  3.0  5.1   
virginica   2.974    3.0  0.322497  ...            5.55  0.551895  4.5  6.9   

           petal_width_cm                                    
                    count   mean median       std  min  max  
species                                                      
setosa                 50  0.246    0.2  0.105386  0.1  0.6  
versicolor             50  1.326    1.3  0.197753  1.0  1.8  
virginica              50  2.026    2.0  0.274650  1.4  2.5  

[3 rows x 24 columns]

---

# **Referências**

* NEMEC, Fernando. Estatística para Data Science e Inteligência Artificial: os fundamentos estatísticos que você precisa para trabalhar com dados. São Paulo: Nemec Consultoria, 2025.

* Fernando.nemec@gmail.com
* www.nmx.com.br
* 11 98405-4380

---

© 2026 Jones Egydio. Todos os direitos reservados.  

Conecte-se comigo no [LinkedIn](https://www.linkedin.com/in/jones-egydio-msc-3300359/).
